### Preprocessing and Exploration (Option A Dataset)

In [ ]:
RUN_BASELINE = 1
RUN_PLM = 1

import logging

logging.basicConfig(level=logging.ERROR)

In [ ]:
import time
import json
import pandas as pd


data_path = "/kaggle/input/fake-and-real-news-dataset"

fake = pd.read_csv(f"{data_path}/Fake.csv")
real = pd.read_csv(f"{data_path}/True.csv")

print("Fake:", fake.shape)
print("Real:", real.shape)

In [ ]:
print(fake["subject"].value_counts())

In [ ]:
print(real["subject"].value_counts())

In [ ]:
fake["label"] = 0
real["label"] = 1

df = pd.concat([fake, real], ignore_index=True)
df.drop(columns=["date"], inplace=True)

print(df.shape)
print(df["label"].value_counts())

In [ ]:
df.drop(columns=["subject"], inplace=True)

df["title"] = df["title"].fillna("")
df["text"] = df["text"].fillna("")

df["content"] = df["title"] + " " + df["text"]

df["content"] = (
    df["content"]
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

df = df[["content", "label"]]

In [ ]:
print("Before dropping duplicates:", df.shape)
df = df.drop_duplicates(subset=["content"])
print("After dropping duplicates:", df.shape)
print(df["label"].value_counts())

### Train / Test Split

In [ ]:
from sklearn.model_selection import train_test_split

X = df["content"]
y = df["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

### Partie A - TF-IDF + Logistic Regression

In [ ]:
import json
import pandas as pd
import time
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

if RUN_BASELINE:
    vectorizer = TfidfVectorizer(
        lowercase=True,
        stop_words="english",
        ngram_range=(1, 2),
        max_features=768
    )
    
    X_train_tfidf = vectorizer.fit_transform(X_train)
    X_test_tfidf = vectorizer.transform(X_test)
    
    model = LogisticRegression(
        max_iter=1000,
        random_state=42
    )

    start_time = time.perf_counter()
    
    model.fit(X_train_tfidf, y_train)

    baseline_training_time = time.perf_counter() - start_time
    
    y_test_pred = model.predict(X_test_tfidf)
    
    baseline_accuracy = accuracy_score(y_test, y_test_pred)
    baseline_f1 = f1_score(y_test, y_test_pred)

    print("=============================== Baseline Model Evaluation ===============================")

    print(f"Baseline Model Accuracy: {baseline_accuracy:.4f}")
    print(f"Baseline Model F1 Score: {baseline_f1:.4f}")
    print(f"Baseline Model Computation Time: {baseline_training_time:.4f} seconds")

    print("=========================================================================================")
    
    cm = confusion_matrix(y_test, y_test_pred)
    
    disp = ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=["Fake", "Real"]
    )
    
    disp.plot(cmap="Blues")
    
    plt.title("Baseline Model Confusion Matrix")
    plt.tight_layout()
    
    plt.savefig(
        "/kaggle/working/baseline_confusion_matrix.png",
        dpi=200
    )
    
    plt.close()
    
    baseline_metrics = {
        "accuracy": round(baseline_accuracy, 4),
        "f1": round(baseline_f1, 4),
        "training_time_seconds": round(baseline_training_time, 4)
    }
    
    with open("/kaggle/working/baseline_metrics.json", "w") as f:
        json.dump(baseline_metrics, f, indent=4)

### Partie B - DistilBERT Encoder + Logistic Regression

In [ ]:
from transformers import AutoTokenizer, AutoModel
import torch
import numpy as np

if RUN_PLM:

    device = torch.device(
        "cuda" if torch.cuda.is_available() else "cpu"
    )

    print("Device:", device)

    if device.type == "cuda":
        print("GPU:", torch.cuda.get_device_name(0))

    model_name = "distilbert-base-uncased"

    tokenizer = AutoTokenizer.from_pretrained(model_name)

    model = AutoModel.from_pretrained(model_name)

    model = model.to(device)

    model.eval()
    model.requires_grad_(False)


def mean_pooling(model_output, attention_mask):

    token_embeddings = model_output.last_hidden_state

    input_mask_expanded = (
        attention_mask
        .unsqueeze(-1)
        .expand(token_embeddings.size())
        .float()
    )

    sum_embeddings = torch.sum(
        token_embeddings * input_mask_expanded,
        dim=1
    )

    sum_mask = torch.clamp(
        input_mask_expanded.sum(dim=1),
        min=1e-9
    )

    return sum_embeddings / sum_mask

In [ ]:
def extract_embeddings(
    texts,
    tokenizer,
    model,
    device,
    batch_size=16,
    max_length=256,
):
    """
    Convert a collection of texts into DistilBERT embeddings.

    Each article is converted into one 768-dimensional vector.
    """

    all_embeddings = []

    model.eval()

    for start in range(0, len(texts), batch_size):

        # Get one batch of articles
        batch_texts = texts[
            start:start + batch_size
        ].tolist()

        # Tokenize the batch
        inputs = tokenizer(
            batch_texts,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors="pt",
        )

        # Move input tensors to GPU or CPU
        inputs = {
            key: value.to(device)
            for key, value in inputs.items()
        }

        # No gradients are needed because DistilBERT is frozen
        with torch.no_grad():

            # Run DistilBERT
            outputs = model(**inputs)

            # Convert token embeddings into
            # one embedding per article
            embeddings = mean_pooling(
                outputs,
                inputs["attention_mask"]
            )

        # Move embeddings back to CPU
        # and convert them to NumPy arrays
        all_embeddings.append(
            embeddings.cpu().numpy()
        )

    # Combine all batches
    return np.concatenate(
        all_embeddings,
        axis=0
    )

In [ ]:
if RUN_PLM:

    print("\nExtracting training embeddings...")

    start_time = time.perf_counter()

    X_train_plm = extract_embeddings(
        texts=X_train,
        tokenizer=tokenizer,
        model=model,
        device=device,
    )

    print(
        "Training embedding shape:",
        X_train_plm.shape
    )

In [ ]:
if RUN_PLM:

    print("\nExtracting test embeddings...")

    X_test_plm = extract_embeddings(
        texts=X_test,
        tokenizer=tokenizer,
        model=model,
        device=device,
    )

    print(
        "Test embedding shape:",
        X_test_plm.shape
    )

In [ ]:
if RUN_PLM:

    print("\nTraining Logistic Regression...")

    plm_classifier = LogisticRegression(
        max_iter=1000,
        random_state=42
    )

    plm_classifier.fit(X_train_plm, y_train)

    plm_training_time = time.perf_counter() - start_time

    y_test_plm_pred = plm_classifier.predict(X_test_plm)

    plm_accuracy = accuracy_score(y_test, y_test_plm_pred)

    plm_f1 = f1_score(y_test, y_test_plm_pred)

    print("=============================== Frozen PLM Model Evaluation ===============================")
    print(f"Frozen PLM Model Accuracy: {plm_accuracy:.4f}")
    print(f"Frozen PLM Model F1 Score: {plm_f1:.4f}")
    print(f"Frozen PLM Model Training Time (Including Feature Extraction with DistilBERT): {plm_training_time:.4f} seconds")
    print("===========================================================================================")

    plm_cm = confusion_matrix(y_test, y_test_plm_pred)

    plm_disp = ConfusionMatrixDisplay(confusion_matrix=plm_cm, display_labels=["Fake", "Real"])

    plm_disp.plot(cmap="Reds")
    plt.title("Frozen DistilBERT Confusion Matrix")

    plt.tight_layout()

    plt.savefig("/kaggle/working/frozen_plm_confusion_matrix.png", dpi=200, bbox_inches="tight")

    plt.close()

    frozen_plm_metrics = {
        "accuracy": round(plm_accuracy, 4),
        "f1": round(plm_f1, 4),
        "training_time_seconds": round(plm_training_time, 4),
    }

    with open("/kaggle/working/frozen_plm_metrics.json", "w") as f:
        json.dump(frozen_plm_metrics, f, indent=4)

    plt.close()
